In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

df = pd.read_parquet("sales_features.parquet").reset_index()

# Define the percentage of stores to select (e.g., 10%)
percentage = 0.05
# Get unique store_ids
unique_stores = df['store'].unique()

# Calculate the number of stores to select
num_stores = int(len(unique_stores) * percentage)

# Randomly select store_ids without replacement
selected_stores = np.random.choice(unique_stores, size=num_stores, replace=False)

# Filter the DataFrame to include only selected stores
df = df[df['store'].isin(selected_stores)].copy()

# Optional: Reset index if needed
df.reset_index(drop=True, inplace=True)

# Create a unique store_id
df['store_id'] = df.groupby(['name', 'address', 'city', 'zipcode', 'county']).ngroup()

# Encode categorical features
categorical_features = ['store', 'city', 'county', 'holiday_name']
label_encoders = {}
for feature in categorical_features:
    le = LabelEncoder()
    df[feature] = le.fit_transform(df[feature].astype(str))
    label_encoders[feature] = le

# Normalize numerical features
numeric_features = ['sale_dollars', 'lon', 'lat', 'store_size', 'sale_dollars_lag_1d']
scaler = StandardScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])

# Split data temporally
df = df.sort_values(['store', 'date'])
train_df = df[df['date'] < '2023-01-01']  # Adjust cutoff
test_df = df[df['date'] >= '2023-01-01']

In [7]:
def create_sequences(data, seq_length, pred_length):
    sequences, targets = [], []
    for store_id in data['store_id'].unique():
        store_data = data[data['store_id'] == store_id].sort_values('date')
        n = len(store_data)
        for i in range(n - seq_length - pred_length + 1):
            seq = store_data.iloc[i:i+seq_length]
            target = store_data.iloc[i+seq_length:i+seq_length+pred_length]['sale_dollars'].values
            sequences.append(seq)
            targets.append(target)
    return np.array(sequences), np.array(targets)

seq_length, pred_length = 30, 7
X_train, y_train = create_sequences(train_df, seq_length, pred_length)
X_test, y_test = create_sequences(test_df, seq_length, pred_length)

In [12]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense, LSTM, Concatenate, Attention, LayerNormalization
from tensorflow.keras.models import Model

# Static inputs
static_cat_inputs = [Input(shape=(1,), name=f'static_cat_{feat}') for feat in ['store', 'city', 'county']]
static_real_inputs = [Input(shape=(1,), name=f'static_real_{feat}') for feat in ['lon', 'lat', 'store_size']]

time_varying_known_features = []
time_varying_observed_features = []

# Time-varying inputs
time_known_input = Input(shape=(seq_length, len(time_varying_known_features)), name='time_known')
time_observed_input = Input(shape=(seq_length, len(time_varying_observed_features)), name='time_observed')

# Static embeddings
static_embs = [Embedding(input_dim=df[feat].nunique(), output_dim=5)(inp) for feat, inp in zip(['store', 'city', 'county'], static_cat_inputs)]
static_embs = [tf.squeeze(emb, axis=1) for emb in static_embs]
static_concat = Concatenate()(static_embs + static_real_inputs)
static_concat = Dense(16)(static_concat)

# Temporal processing
lstm_out = LSTM(64, return_sequences=True)(time_observed_input)
attn_out = Attention()([lstm_out, lstm_out])
temporal_concat = Concatenate()([time_known_input, attn_out])

# Combine static and temporal
static_expanded = tf.keras.layers.RepeatVector(seq_length)(static_concat)
combined = Concatenate()([static_expanded, temporal_concat])

# Decoder and output
decoder_out = LSTM(64)(combined)
output = Dense(pred_length)(decoder_out)

model = Model(inputs=static_cat_inputs + static_real_inputs + [time_known_input, time_observed_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

ValueError: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.ops`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```
